In [1]:
# from utility.plot_spectrum import plot_spectrum
# from utility.adjust_redshift import adjust_redshift
# from utility.my_colors import palette_light, pink

# from processing.remove_spikes import remove_spikes
# from processing.fit_continuum import fit_continuum
# from processing.quality_cuts import quality_cuts
# from processing.make_bal_mask import make_bal_mask
# from processing.fit_gaussians import fit_single_gaussian, fit_two_gaussians, ftest_which_gaussian
# from processing.measure_line import measure_line
# from processing.parameters import CIV_CONTINUUM_WINDOWS
# from processing.get_absorption_intervals import get_absorption_intervals

# from data.local.hamann_erq_desi_idxs import valid_sdss_names

# from sparcl.client import SparclClient
# from dl import queryClient as qc
# from astropy.table import Table
# from astropy.convolution import convolve, Gaussian1DKernel
# from scipy.interpolate import interp1d
# from datetime import datetime
# from scipy.signal import argrelmin, argrelmax
# from ast import literal_eval

# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# import pandas as pd
# import warnings
# import json
# import os

# hamann_fits_path = "data/local/ERQs_core_all_MNRAS.fits"
# search_log_path = "data/local/hamann-objects-desi-search-log.json"
# matched_desi_spectra_path = "data/local/matched_desi_spectra.json"

from utility.plot_spectrum import plot_spectrum
from utility.my_colors import palette_dark, palette_light
from utility.adjust_redshift import adjust_redshift

from processing.remove_spikes import remove_spikes
from processing.fit_continuum import fit_continuum
from processing.quality_cuts import quality_cuts
from processing.get_absorption_intervals import get_absorption_intervals
from processing.fit_gaussians import fit_single_gaussian, fit_two_gaussians, ftest_which_gaussian
from processing.measure_line import measure_line
from processing.parameters import CIV_CONTINUUM_WINDOWS, CIV_FIT_WINDOW, NV_LYA_CONTINUUM_WINDOWS, LYA_NV_FIT_WINDOW, CIV_AIR, LYA_AIR, NV_AIR, C_KMS
from processing.scale_civ_to_nv import scale_civ_to_nv

from astropy.table import Table

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


found_desi_spectra_stats_path = "data/found_desi_spectra_stats.json"


### preprocess

In [2]:
# hamann_data = Table.read("data/local/ERQs_core_all_MNRAS.fits").to_pandas() #pd.read_csv(hamann_erq_path,index_col=0)

# hamann_data["sdss_name"] = hamann_data["sdss_name"].astype("string")

# match_df = pd.read_json(search_log_path)
# match_df = match_df[["SDSS_id","best_match_id"]].copy()
# match_df["SDSS_id"] = match_df["SDSS_id"].str[1:]

# match_df = match_df.rename(columns={"best_match_id":"targetid", "SDSS_id":"sdss_name"})

# hamann_erq = hamann_data.merge(match_df,on="sdss_name",how="left")

# hamann_erq.info()

In [3]:
# client = SparclClient()

# idxs = [int(x) for x in hamann_erq["targetid"] if x != "Not found"]

# output_fields = ['specid', 'redshift', 'wavelength', 'flux', 'ivar', 'spectype', 'targetid', 'redshift_warning']

# spectra = client.retrieve_by_specid(idxs, dataset_list=["DESI-DR1"], include=output_fields)

In [4]:
# spectra_df = pd.DataFrame(spectra.data[1:])

# spectra_df.to_json(matched_desi_spectra_path,default_handler=str)

# spectra_df

### measure lines

In [5]:
# spectra_df = pd.read_json(matched_desi_spectra_path)

# stats_half = hamann_erq[['sdss_name', 'rew', 'fwhm', 'kt80', 'targetid']].copy()

# spectra_df["targetid"] = spectra_df["targetid"].astype("string")

# data = spectra_df.merge(stats_half,on="targetid")

# data = data[data["sdss_name"].isin(valid_sdss_names)]

# data = data.drop_duplicates(subset="sdss_name")

# data.to_json(found_desi_spectra_stats_path)

In [6]:
data = pd.read_json(found_desi_spectra_stats_path)
data = data.reset_index(drop=True)

# set up stats dataframe
stats = Table.read("data/local/ERQs_core_all_MNRAS.fits").to_pandas()
stats["sdss_name"] = stats["sdss_name"].astype("string")
stats = stats[["sdss_name","rew","rew_nv","fwhm","kt80"]].copy()
stats = stats.set_index("sdss_name")
stats = stats.rename(columns={"rew":"civ_rew_hamann","rew_nv":"nv_rew_hamann","fwhm":"civ_fwhm_hamann","kt80":"civ_kt80_hamann"})

stats["civ_fwhm_mine"] = None
stats["civ_kt80_mine"] = None
stats["civ_rew_mine"], stats["nv_rew_mine"], stats["lya_rew_mine"] = None, None, None
stats["civ_fit_rew_mine"], stats["nv_fit_rew_mine"], stats["lya_fit_rew_mine"] = None, None, None

In [7]:
def test(idx):    
    lam = np.array(data["wavelength"][idx])
    flux = np.array(data["flux"][idx])
    ivar = np.array(data["ivar"][idx])
    z = data["redshift"][idx]


    ## ADJUST REDSHIFT
    lam, flux, ivar = adjust_redshift(z, lam, flux, ivar=ivar)


    ## EXTRACT VALID PIXELS & REMOVE OUTLIERS

    # mask bad pixels, i.e. pixels where flux is zero, ivar is zero or infinity
    valid_pixel_mask = ((lam > 1150) & (lam < 1810)) & (ivar > 0) & np.isfinite(ivar) & (flux != 0)

    # apply mask
    lam = lam[valid_pixel_mask]
    flux = flux[valid_pixel_mask]
    ivar = ivar[valid_pixel_mask]

    # create noise
    noise = np.abs(1/np.sqrt(ivar))

    # remove spikes
    flux_clean, _ = remove_spikes(lam,flux,ivar=ivar)


    ## FIT CONTINUUM BELOW CIV
    result_civ_cont, continuum_civ, _ = fit_continuum(lam, flux_clean, ivar=ivar, windows=CIV_CONTINUUM_WINDOWS, lambda_ref=1700.0)


    ## QUALITY CUTS (based on CIV region)

    # TODO: fix all this, like i think it doesnt do shit currently and you need to get the values fixed
    rejected, reason, flags = quality_cuts(lam, flux_clean, continuum_civ, result_civ_cont, max_slope=10.0)

    if rejected:
        print(f'Rejected because: {reason}')
        
    else:
        ## FIT CIV LINE

        # subtract CIV continuum
        flux_sub_civ = flux_clean - continuum_civ if not rejected else None

        # get absorption mask
        _, civ_absorption_mask = get_absorption_intervals(lam,flux_sub_civ,noise,continuum_civ, window=CIV_FIT_WINDOW)
        # civ_absorption_mask = (flux_sub_civ < -noise)

        # fit single gaussian, use result as parameter guesses for two gaussian fit
        result1_civ, profile1_civ = fit_single_gaussian(lam, flux_sub_civ, ivar=ivar, mask=civ_absorption_mask, window=CIV_FIT_WINDOW)
        result2_civ, profile2_civ = fit_two_gaussians(lam, flux_sub_civ, ivar=ivar, mask=civ_absorption_mask, single_result=result1_civ, window=CIV_FIT_WINDOW)

        # F-test which fit is better
        accept_two_civ, p_val_civ = ftest_which_gaussian(result1_civ, result2_civ)

        # depending on F-test result, assign respective profile and best parameter dictionary to general variables
        profile_civ = profile2_civ if accept_two_civ else profile1_civ
        params_dict_civ = result2_civ.params.valuesdict() if accept_two_civ else result1_civ.params.valuesdict()


        ## CONTINUUM BELOW NV AND LYA
        _, continuum_nv_lya, _ = fit_continuum(lam, flux_clean, ivar=ivar, windows=NV_LYA_CONTINUUM_WINDOWS, lambda_ref=1290.0)  # fit continuum
        flux_sub_nv_lya = flux_clean - continuum_nv_lya  # subtract continuum


        ## NV PROFILE

        # make LYA mask
        # idea: mask left and right by a certain fixed window (left: 60% of lam(NV)-lam(LYA), right 40%),
        # then shift by the difference between CIV fit peak and air wavelengths to make it more robust in case of bad redshift values
        diff_civ = lam[np.argmax(profile_civ)] - CIV_AIR
        m = (NV_AIR-LYA_AIR)
        c = LYA_AIR + diff_civ
        lims = (c-0.6*m,c+0.5*m)
        lya_mask = (lam > lims[0]) & (lam < lims[1])
        
        # fit NV by scaling CIV template
        _, profile_nv = scale_civ_to_nv(lam,flux_sub_nv_lya,params_dict_civ,accept_two_civ,ivar=ivar,mask=lya_mask)
        

        ## LYA PROFILE

        # subtract NV profile
        flux_sub_lya = flux_sub_nv_lya - profile_nv

        # get absorption mask
        _, lya_absorption_mask = get_absorption_intervals(lam,flux_sub_lya,continuum_nv_lya,noise,window=LYA_NV_FIT_WINDOW)  # first is intervals for plotting

        # fit single gaussian, use result as parameter guesses for two gaussian fit
        result1_lya, profile1_lya = fit_single_gaussian(lam, flux_sub_lya, ivar=ivar, mask=lya_absorption_mask, window=LYA_NV_FIT_WINDOW)
        result2_lya, profile2_lya = fit_two_gaussians(lam, flux_sub_lya, ivar=ivar, mask=lya_absorption_mask, single_result=result1_lya, window=LYA_NV_FIT_WINDOW)

        # F-test which fit is better, assign to general variable
        accept_two_lya, p_val_lya = ftest_which_gaussian(result1_lya, result2_lya)
        profile_lya = profile2_lya if accept_two_lya else profile1_lya


        ## LINE MEASUREMENT

        # CIV
        line_stats_civ, civ_integration_win = measure_line(lam, profile_civ, flux_sub_civ, continuum_civ, window=CIV_FIT_WINDOW,get_window=True) 

        # NV
        flux_sub_nv = flux_sub_nv_lya - profile_lya  # subtract Lya profile
        line_stats_nv, nv_integration_win = measure_line(lam,profile_nv,flux_sub_nv,continuum_nv_lya,window=LYA_NV_FIT_WINDOW,get_window=True)

        # Lya
        line_stats_lya, lya_integration_win = measure_line(lam,profile_lya,flux_sub_lya,continuum_nv_lya,window=LYA_NV_FIT_WINDOW,get_window=True)
        

        ## RESULTS

        # extract
        sdss_name = data["sdss_name"][idx]

        rew_civ = line_stats_civ["ew_aa"]
        rew_nv = line_stats_nv["ew_aa"]
        rew_lya = line_stats_lya["ew_aa"]

        # print
        print(f'''
            SDSS: {sdss_name}

            Lya:\t{rew_lya:.3f}
            NV:\t\t{rew_nv:.3f}\t{stats.loc[sdss_name,"nv_rew_hamann"]:.3f} (Hamann)
            CIV:\t{rew_civ:.3f}\t{stats.loc[sdss_name,"civ_rew_hamann"]:.3f} (Hamann)
            
            compared to CIV:
            Lya\t\tNV\tCIV
            {(rew_lya/rew_civ):.2f}\t{(rew_nv/rew_civ):.2f}\t1
        ''')

        # to dataframe
        stats.loc[sdss_name,"civ_rew_mine"] = rew_civ
        stats.loc[sdss_name,"nv_rew_mine"] = rew_nv
        stats.loc[sdss_name,"lya_rew_mine"] = rew_lya
        
        stats.loc[sdss_name,"civ_fit_rew_mine"] = line_stats_civ["fit_rew"]
        stats.loc[sdss_name,"nv_fit_rew_mine"] = line_stats_nv["fit_rew"]
        stats.loc[sdss_name,"lya_fit_rew_mine"] = line_stats_lya["fit_rew"]

        stats.loc[sdss_name,"civ_fwhm_mine"] = line_stats_civ["fwhm_kms"]
        stats.loc[sdss_name,"civ_kt80_mine"] = line_stats_civ["kt80"]


        # ## PLOTTING

        # plot_mask_lya_nv = (lam > LYA_NV_FIT_WINDOW[0]) & (lam < LYA_NV_FIT_WINDOW[1])

        # if nv_integration_win is not None:
        #     plot_spectrum(lam[plot_mask_lya_nv],flux_sub_nv_lya[plot_mask_lya_nv],
        #         # line profiles
        #         additional_flux=[profile_lya[plot_mask_lya_nv],profile_nv[plot_mask_lya_nv]],
        #         additional_flux_label=[r"Ly$\alpha$ fit","NV fit"],
        #         additional_flux_color=["#4b0082","#007090"],
        #         # absorption
        #         scatter_x=lam[lya_absorption_mask],
        #         scatter_y=flux_sub_nv_lya[lya_absorption_mask],
        #         scatter_label="absorption detected",
        #         # integration regions
        #         shaded_regions=[lya_integration_win,nv_integration_win,lims],
        #         shaded_regions_colors=["#d9ace3","#adcae3","grey"],
        #         shaded_regions_label=[r"Ly$\alpha$ integration window","NV integration window",r"Ly$\alpha$ mask"],
        #         # lines
        #         line_lambda=[LYA_AIR,NV_AIR],
        #         line_label=[r"Ly$\alpha$","NV"],
        #         line_color=palette_dark[1:3],
        #         # general
        #         line_width_factor=2
        #     )
        # else:
        #     plot_spectrum(lam[plot_mask_lya_nv],flux_sub_nv_lya[plot_mask_lya_nv],
        #         # line profiles
        #         additional_flux=[profile_lya[plot_mask_lya_nv],profile_nv[plot_mask_lya_nv]],
        #         additional_flux_label=[r"Ly$\alpha$ fit","NV fit"],
        #         additional_flux_color=["#4b0082","#007090"],
        #         # absorption
        #         scatter_x=lam[lya_absorption_mask],
        #         scatter_y=flux_sub_nv_lya[lya_absorption_mask],
        #         scatter_label="absorption detected",
        #         # integration regions
        #         shaded_regions=[lya_integration_win,lims],
        #         shaded_regions_colors=["#d9ace3","grey"],
        #         shaded_regions_label=[r"Ly$\alpha$ integration window",r"Ly$\alpha$ mask"],
        #         # lines
        #         line_lambda=[LYA_AIR,NV_AIR],
        #         line_label=[r"Ly$\alpha$","NV"],
        #         line_color=palette_dark[1:3],
        #         # general
        #         line_width_factor=2
        #     )
        

        # plot_mask_civ = (lam > CIV_FIT_WINDOW[0]) & (lam < CIV_FIT_WINDOW[1])

        # plot_spectrum(lam[plot_mask_civ],flux_sub_civ[plot_mask_civ],
        #     # line profiles
        #     additional_flux=[profile_civ[plot_mask_civ]],
        #     additional_flux_label=["CIV fit"],
        #     additional_flux_color=["#6b8e23"],
        #     # absorption
        #     scatter_x=lam[civ_absorption_mask],
        #     scatter_y=flux_sub_civ[civ_absorption_mask],
        #     scatter_label="absorption detected",
        #     # integration regions
        #     shaded_regions=[civ_integration_win],
        #     shaded_regions_colors=["#d3ff8d"],
        #     # lines
        #     line_lambda=[CIV_AIR],
        #     line_label=["CIV"],
        #     line_color=[palette_dark[3]],
        #     # general
        #     line_width_factor=2
        # )

In [8]:
for i in range(len(data)):
    print("---------------------------------------\n---------------------------------------")
    test(i)

---------------------------------------
---------------------------------------

            SDSS: 233636.99+065231.0

            Lya:	200.100
            NV:		215.417	124.533 (Hamann)
            CIV:	171.310	128.271 (Hamann)
            
            compared to CIV:
            Lya		NV	CIV
            1.17	1.26	1
        
---------------------------------------
---------------------------------------

            SDSS: 082649.30+163945.2

            Lya:	68.135
            NV:		42.040	99.313 (Hamann)
            CIV:	97.301	204.888 (Hamann)
            
            compared to CIV:
            Lya		NV	CIV
            0.70	0.43	1
        
---------------------------------------
---------------------------------------

            SDSS: 083448.48+015921.1

            Lya:	1066.791
            NV:		208.674	305.821 (Hamann)
            CIV:	249.484	209.440 (Hamann)
            
            compared to CIV:
            Lya		NV	CIV
            4.28	0.84	1
        
----------------------

In [12]:
stats = stats.dropna(subset="civ_rew_mine")

current_stats_path = "data/local/desi_current_best_measurements.csv"

stats.to_csv(current_stats_path)

stats

,civ_rew_hamann,nv_rew_hamann,civ_fwhm_hamann,civ_kt80_hamann,civ_fwhm_mine,civ_kt80_mine,civ_rew_mine,nv_rew_mine,lya_rew_mine,civ_fit_rew_mine,nv_fit_rew_mine,lya_fit_rew_mine,civ_rew_abs_diff,nv_rew_abs_diff,civ_rew_rel_diff,nv_rew_rel_diff
sdss_name,,,,,,,,,,,,,,,,
000746.19+122223.9,219.633801,64.601598,2432.830081,0.284162,2547.325820,0.311353,271.322034,38.366665,641.855413,266.528800,71.023123,671.887679,51.688233,-26.234934,0.235338,0.406103
002400.25+245031.9,143.689957,205.776721,3523.281699,0.372201,3478.581796,0.298861,132.981058,203.242732,109.427097,133.979147,228.942288,132.689836,-10.708899,-2.533989,0.074528,0.012314
014111.13-031852.5,101.008201,157.859652,2985.648457,0.372200,2842.545900,0.355123,110.477415,150.898459,270.440536,109.642978,122.432754,257.037855,9.469214,-6.961194,0.093747,0.044097
020932.15+312202.7,107.676570,103.792779,2180.064301,0.344675,2916.152923,0.334114,125.976874,123.025906,363.875327,134.250465,108.966192,356.438747,18.300304,19.233127,0.169956,0.185303
080547.66+454159.0,108.689884,140.261594,2667.070587,0.210800,3307.016523,0.280357,138.463625,293.814895,400.641715,140.376361,293.877040,464.301877,29.773742,153.553300,0.273933,1.094764
082536.31+200040.3,210.804566,52.193434,3265.174575,0.171451,4054.446073,0.219185,96.645606,12.424330,68.606412,91.836766,14.972355,70.231285,-114.158960,-39.769104,0.541539,0.761956
082649.30+163945.2,204.888419,99.312833,5633.198558,0.329619,4217.192405,0.229549,97.301380,42.039555,68.134831,94.897614,49.409686,74.390688,-107.587038,-57.273278,0.525101,0.576696
082653.42+054247.3,204.619151,324.365085,2433.821835,0.357933,2385.829881,0.310959,198.035231,251.304363,323.762341,196.910308,239.529945,363.352424,-6.583921,-73.060722,0.032176,0.225242
083448.48+015921.1,209.439757,305.821165,2863.092351,0.364786,2887.103165,0.361463,249.484305,208.673610,1066.791354,248.109475,211.425406,1112.594105,40.044548,-97.147555,0.191198,0.317661


In [10]:
# stats = data[data["my_rew"].notnull()][['sdss_name','redshift', 'rew', 'fit_rew', 'fwhm', 'kt80', 'my_fwhm', 'my_rew', 'my_kt80']]

stats[['civ_fwhm_mine', 'civ_kt80_mine', 'civ_rew_mine', 'nv_rew_mine',
       'lya_rew_mine', 'civ_fit_rew_mine', 'nv_fit_rew_mine',
       'lya_fit_rew_mine']] = stats[['civ_fwhm_mine', 'civ_kt80_mine', 'civ_rew_mine', 'nv_rew_mine',
       'lya_rew_mine', 'civ_fit_rew_mine', 'nv_fit_rew_mine',
       'lya_fit_rew_mine']].astype(float)

stats["civ_rew_abs_diff"] = stats["civ_rew_mine"] - stats["civ_rew_hamann"]
stats["nv_rew_abs_diff"] = stats["nv_rew_mine"] - stats["nv_rew_hamann"]


stats["civ_rew_rel_diff"] = abs(stats["civ_rew_abs_diff"] / stats["civ_rew_hamann"])
stats["nv_rew_rel_diff"] = abs(stats["nv_rew_abs_diff"] / stats["nv_rew_hamann"])

stats[['civ_rew_abs_diff', 'nv_rew_abs_diff',
       'civ_rew_rel_diff', 'nv_rew_rel_diff']].describe()

,civ_rew_abs_diff,nv_rew_abs_diff,civ_rew_rel_diff,nv_rew_rel_diff
count,30.000000,30.000000,30.000000,30.000000
mean,-10.069848,-13.153474,0.202116,0.358425
std,56.125873,77.723235,0.163339,0.349307
min,-161.086425,-231.277521,0.014188,0.004798
25%,-21.368451,-39.384726,0.084129,0.119972
50%,-8.242548,-24.803009,0.156473,0.258331
75%,14.328261,28.392625,0.280544,0.389675
max,140.726980,153.553300,0.549729,1.305929
